# Quick-Commerce Delivery Intelligence — Manthan Hackathon

## 1. Project Introduction

This notebook addresses **Problem Statement 1: Data Quality & Reliability only**. It checks the source data, documents issues and treatments, preserves useful orders, and writes an auditable cleaned dataset and quality report. Missing fields are not filled with invented values, questionable coordinates are marked missing, and statistically unusual delivery durations are retained unless they are impossible or unusable as a target. No model training, SLA definition, feature engineering, or broader EDA is performed.

## 2. Import Libraries

Use pandas and NumPy for reproducible inspection and validation.

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_colwidth', 120)

## 3. Load Dataset

The notebook searches the current folder and its parent for the required relative path `data/train.csv`. All columns are initially read as strings so missing-value spellings and formatting issues can be assessed before type conversion.

In [2]:
candidate_roots = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((root for root in candidate_roots if (root / 'data' / 'train.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not find data/train.csv relative to the notebook working directory. '
        'Place the source CSV at the project-relative path data/train.csv and rerun.'
    )

INPUT_PATH = PROJECT_ROOT / 'data' / 'train.csv'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
CLEANED_PATH = PROJECT_ROOT / 'data' / 'cleaned_data.csv'
REPORT_PATH = OUTPUT_DIR / 'data_quality_report.csv'
SUMMARY_PATH = OUTPUT_DIR / 'data_quality_summary.txt'

try:
    raw_df = pd.read_csv(INPUT_PATH, dtype='string', keep_default_na=False)
except (OSError, pd.errors.ParserError, UnicodeError) as exc:
    raise RuntimeError(f'Unable to read the dataset at {INPUT_PATH}: {exc}') from exc

print(f'Loaded: {INPUT_PATH.relative_to(PROJECT_ROOT)}')
print(f'Rows: {len(raw_df):,} | Columns: {raw_df.shape[1]}')

Loaded: data\train.csv
Rows: 45,593 | Columns: 20


## 4. Initial Dataset Exploration

This is basic structure inspection only: sample rows, dimensions, column names, data types, and a complete `DataFrame.info()` report.

In [3]:
original_shape = raw_df.shape
print('First five rows:')
display(raw_df.head())
print(f'Shape: {raw_df.shape}')
print('Column names:')
print(raw_df.columns.tolist())
print('Data types on initial string-preserving load:')
display(raw_df.dtypes.rename('dtype').to_frame())
print('Complete dataset information:')
raw_df.info(show_counts=True)

First five rows:


,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,conditions Sunny,High,2,Snack,motorcycle,0,No,Urban,(min) 24
1,0xb379,BANGRES18DEL02,34,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,conditions Stormy,Jam,2,Snack,scooter,1,No,Metropolitian,(min) 33
2,0x5d6d,BANGRES19DEL01,23,4.4,12.914264,77.6784,12.924264,77.6884,19-03-2022,08:30:00,08:45:00,conditions Sandstorms,Low,0,Drinks,motorcycle,1,No,Urban,(min) 26
3,0x7a6a,COIMBRES13DEL02,38,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,conditions Sunny,Medium,0,Buffet,motorcycle,1,No,Metropolitian,(min) 21
4,0x70a2,CHENRES12DEL01,32,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,conditions Cloudy,High,1,Snack,scooter,1,No,Metropolitian,(min) 30


Shape: (45593, 20)
Column names:
['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken(min)']
Data types on initial string-preserving load:


,dtype
ID,string
Delivery_person_ID,string
Delivery_person_Age,string
Delivery_person_Ratings,string
Restaurant_latitude,string
Restaurant_longitude,string
Delivery_location_latitude,string
Delivery_location_longitude,string
Order_Date,string
Time_Orderd,string


Complete dataset information:
<class 'pandas.DataFrame'>
RangeIndex: 45593 entries, 0 to 45592
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   ID                           45593 non-null  string
 1   Delivery_person_ID           45593 non-null  string
 2   Delivery_person_Age          45593 non-null  string
 3   Delivery_person_Ratings      45593 non-null  string
 4   Restaurant_latitude          45593 non-null  string
 5   Restaurant_longitude         45593 non-null  string
 6   Delivery_location_latitude   45593 non-null  string
 7   Delivery_location_longitude  45593 non-null  string
 8   Order_Date                   45593 non-null  string
 9   Time_Orderd                  45593 non-null  string
 10  Time_Order_picked            45593 non-null  string
 11  Weatherconditions            45593 non-null  string
 12  Road_traffic_density         45593 non-null  string
 13  Vehicle_cond

## 5. Missing Value Analysis

Blank/whitespace-only cells and common text sentinels are treated as missing. The weather value `conditions NaN` is also recognized as missing. These values are standardized to `pd.NA`; no records are discarded or values imputed at this stage.

In [4]:
MISSING_TOKENS = {'', 'nan', 'null', 'none', 'na', 'n/a'}

def normalize_missing_values(frame):
    """Trim text and normalize common missing-value tokens without imputing."""
    result = frame.copy()
    for column in result.columns:
        values = result[column].astype('string').str.strip()
        missing = values.str.casefold().isin(MISSING_TOKENS)
        if column == 'Weatherconditions':
            missing |= values.str.casefold().str.fullmatch(r'conditions\s+(nan|null|none|na|n/a)', na=False)
        result[column] = values.mask(missing, pd.NA)
    return result

normalized_df = normalize_missing_values(raw_df)
missing_before = normalized_df.isna().sum().rename('missing_count').to_frame()
missing_before['missing_percent'] = (missing_before['missing_count'] / len(normalized_df) * 100).round(2)
display(missing_before.sort_values('missing_count', ascending=False))
print(f'Total missing cells after sentinel recognition: {int(normalized_df.isna().sum().sum()):,}')

,missing_count,missing_percent
Delivery_person_Ratings,1908,4.18
Delivery_person_Age,1854,4.07
Time_Orderd,1731,3.80
City,1200,2.63
multiple_deliveries,993,2.18
Weatherconditions,616,1.35
Road_traffic_density,601,1.32
Festival,228,0.50
ID,0,0.00
Delivery_person_ID,0,0.00


Total missing cells after sentinel recognition: 9,131


## 6. Duplicate Analysis

Exact duplicates are checked across the complete source rows before any cleaning. Only exact duplicate rows are candidates for removal; similar orders are not treated as duplicates.

In [5]:
duplicate_mask = raw_df.duplicated(keep='first')
duplicate_count = int(duplicate_mask.sum())
print(f'Exact duplicate records beyond the first copy: {duplicate_count:,}')
if duplicate_count:
    display(raw_df.loc[duplicate_mask].head())
else:
    print('No exact duplicate rows were found.')

Exact duplicate records beyond the first copy: 0
No exact duplicate rows were found.


## 7. Rider Age Validation

For consistency across the project, rider age values are considered valid for analysis when they fall between 18 and 80 years, inclusive. This is a data-validation rule and does not represent rider professional experience. Values outside this inclusive range are invalid for age-based analysis. Invalid and unparseable values are marked for audit and set missing in the pre-imputation cleaned field; the order is retained. The original source text and row-level flags are kept in the cleaned output.

In [6]:
AGE_MIN, AGE_MAX = 18, 80
AGE_EXPLANATION = 'For consistency across the project, rider age values are considered valid for analysis when they fall between 18 and 80 years, inclusive. This is a data-validation rule and does not represent rider professional experience.'
age_values = pd.to_numeric(normalized_df['Delivery_person_Age'], errors='coerce')
age_invalid_mask = age_values.notna() & ~age_values.between(AGE_MIN, AGE_MAX, inclusive='both')
age_unparseable_mask = normalized_df['Delivery_person_Age'].notna() & age_values.isna()
print(f'Age validation range: {AGE_MIN}-{AGE_MAX} years, inclusive.')
print(f'Invalid ages outside {AGE_MIN}-{AGE_MAX}: {int(age_invalid_mask.sum()):,}')
print(f'Non-missing ages that cannot be parsed numerically: {int(age_unparseable_mask.sum()):,}')
print('Observed numeric age range:', age_values.min(), 'to', age_values.max())
print('Age validation is not a measure of rider professional experience.')
display(raw_df.loc[age_invalid_mask | age_unparseable_mask, ['Delivery_person_Age']].head(10))

Age validation range: 18-80 years, inclusive.
Invalid ages outside 18-80: 38
Non-missing ages that cannot be parsed numerically: 0
Observed numeric age range: 15 to 50
Age validation is not a measure of rider professional experience.


,Delivery_person_Age
2387,15
2905,15
2951,15
5902,15
9156,15
10900,15
11125,15
12093,15
15124,15
15927,15


## 8. Rider Rating Validation

Rider ratings must be numeric and within the stated 1–5 range. Out-of-range and nonnumeric non-missing values are invalid measurements, not reasons to discard the full order; they will be set to missing.

In [7]:
rating_values = pd.to_numeric(normalized_df['Delivery_person_Ratings'], errors='coerce')
rating_invalid_mask = rating_values.notna() & ~rating_values.between(1, 5)
rating_unparseable_mask = normalized_df['Delivery_person_Ratings'].notna() & rating_values.isna()
print(f'Ratings outside 1–5: {int(rating_invalid_mask.sum()):,}')
print(f'Non-missing ratings that cannot be parsed numerically: {int(rating_unparseable_mask.sum()):,}')
print('Observed numeric rating range:', rating_values.min(), 'to', rating_values.max())
display(raw_df.loc[rating_invalid_mask | rating_unparseable_mask, ['Delivery_person_Ratings']].head(10))

Ratings outside 1–5: 53
Non-missing ratings that cannot be parsed numerically: 0
Observed numeric rating range: 1.0 to 6.0


,Delivery_person_Ratings
3586,6
4714,6
5169,6
5362,6
5651,6
6394,6
7031,6
7681,6
9499,6
9535,6


## 9. GPS Coordinate Validation

Coordinates are first checked for numeric values and legal global latitude/longitude bounds. `(0,0)` and paired near-origin values are flagged as placeholders/suspicious for these records (the source delivery coordinates repeatedly use equal latitude and longitude below one degree, including `(0.01,0.01)`). Their coordinate pair will be marked missing—not replaced with guessed locations—while the order is retained. Global bounds are used rather than an undocumented city boundary.

In [8]:
GPS_PAIRS = {
    'restaurant': ('Restaurant_latitude', 'Restaurant_longitude'),
    'delivery': ('Delivery_location_latitude', 'Delivery_location_longitude'),
}
gps_details = {}
for label, (lat_col, lon_col) in GPS_PAIRS.items():
    lat = pd.to_numeric(normalized_df[lat_col], errors='coerce')
    lon = pd.to_numeric(normalized_df[lon_col], errors='coerce')
    nonnumeric = (normalized_df[lat_col].notna() & lat.isna()) | (normalized_df[lon_col].notna() & lon.isna())
    out_of_bounds = lat.notna() & lon.notna() & (~lat.between(-90, 90) | ~lon.between(-180, 180))
    near_origin_pair = lat.notna() & lon.notna() & lat.abs().lt(1) & lon.abs().lt(1) & np.isclose(lat.fillna(np.inf), lon.fillna(-np.inf))
    placeholder = near_origin_pair
    gps_details[label] = {
        'lat': lat, 'lon': lon, 'nonnumeric': nonnumeric,
        'out_of_bounds': out_of_bounds, 'placeholder': placeholder,
        'invalid': nonnumeric | out_of_bounds | placeholder,
    }
    print(
        f"{label.title()}: nonnumeric={int(nonnumeric.sum()):,}, "
        f"out of global bounds={int(out_of_bounds.sum()):,}, "
        f"near-origin placeholder/suspicious pair={int(placeholder.sum()):,}"
    )
restaurant_zero_zero = gps_details['restaurant']['lat'].eq(0) & gps_details['restaurant']['lon'].eq(0)
delivery_001 = gps_details['delivery']['lat'].eq(0.01) & gps_details['delivery']['lon'].eq(0.01)
gps_invalid_records = gps_details['restaurant']['invalid'] | gps_details['delivery']['invalid']
print(f'Records with at least one invalid/suspicious coordinate pair: {int(gps_invalid_records.sum()):,}')
print(f"Restaurant (0,0) rows: {int(restaurant_zero_zero.sum()):,}; delivery (0.01,0.01) rows: {int(delivery_001.sum()):,}")

Restaurant: nonnumeric=0, out of global bounds=0, near-origin placeholder/suspicious pair=3,640
Delivery: nonnumeric=0, out of global bounds=0, near-origin placeholder/suspicious pair=3,640
Records with at least one invalid/suspicious coordinate pair: 3,640
Restaurant (0,0) rows: 3,640; delivery (0.01,0.01) rows: 327


## 10. Date and Timestamp Validation

`Order_Date` is expected in day-month-year form (`%d-%m-%Y`); both order and pickup times are expected as `%H:%M:%S`. Existing missing values are kept distinct from malformed nonempty text. Bad date/time fields will become missing, not cause an otherwise useful order to be deleted.

In [9]:
DATETIME_FORMATS = {
    'Order_Date': '%d-%m-%Y',
    'Time_Orderd': '%H:%M:%S',
    'Time_Order_picked': '%H:%M:%S',
}
parsed_datetimes = {}
date_time_issues = {}
for column, date_format in DATETIME_FORMATS.items():
    values = normalized_df[column]
    parsed = pd.to_datetime(values, format=date_format, errors='coerce')
    malformed = values.notna() & parsed.isna()
    parsed_datetimes[column] = parsed
    date_time_issues[column] = malformed
    print(f'{column}: missing={int(values.isna().sum()):,}; malformed nonempty={int(malformed.sum()):,}')
    if malformed.any():
        print('Examples:', values[malformed].drop_duplicates().head(5).tolist())

Order_Date: missing=0; malformed nonempty=0
Time_Orderd: missing=1,731; malformed nonempty=0
Time_Order_picked: missing=0; malformed nonempty=0


## 11. Delivery Time Cleaning

Extract the numeric target from values such as `(min) 24` using a full-string pattern. Non-numeric, missing, zero, or negative delivery durations cannot be used as a valid outcome and are candidates for row removal; valid durations remain numeric.

In [10]:
TARGET_COLUMN = 'Time_taken(min)'
target_text = normalized_df[TARGET_COLUMN]
target_extracted = target_text.str.extract(r'^\s*\(min\)\s*([+-]?\d+(?:\.\d+)?)\s*$', expand=False)
target_minutes = pd.to_numeric(target_extracted, errors='coerce')
target_invalid_mask = target_minutes.isna() | target_minutes.le(0)
target_parse_failures = target_text.notna() & target_minutes.isna()
print(f'Unparseable non-missing target values: {int(target_parse_failures.sum()):,}')
print(f'Missing, invalid, or nonpositive target values: {int(target_invalid_mask.sum()):,}')
print('Parsed delivery time range (minutes):', target_minutes.min(), 'to', target_minutes.max())
display(pd.DataFrame({'original': target_text.head(), 'numeric_minutes': target_minutes.head()}))

Unparseable non-missing target values: 0
Missing, invalid, or nonpositive target values: 0
Parsed delivery time range (minutes): 10 to 54


,original,numeric_minutes
0,(min) 24,24
1,(min) 33,33
2,(min) 26,26
3,(min) 21,21
4,(min) 30,30


## 12. Outlier Detection

Use the standard 1.5×IQR fences to flag statistical outliers. An IQR flag alone does not prove a record is erroneous. These values are reported and retained; only impossible/nonpositive or unusable target values are removed under the target-validity rule above.

In [11]:
valid_target = target_minutes.dropna()
q1 = valid_target.quantile(0.25)
q3 = valid_target.quantile(0.75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
target_outlier_mask = target_minutes.notna() & ((target_minutes < lower_fence) | (target_minutes > upper_fence))
impossible_time_mask = target_minutes.notna() & target_minutes.le(0)
print(f'Q1={q1:g}, Q3={q3:g}, IQR={iqr:g}; fences={lower_fence:g} to {upper_fence:g} minutes')
print(f'IQR outlier records: {int(target_outlier_mask.sum()):,}')
print(f'Impossible nonpositive delivery times: {int(impossible_time_mask.sum()):,}')
print('Treatment: retain valid IQR outliers and flag them in the report; do not delete them without evidence of error.')

Q1=19, Q3=32, IQR=13; fences=-0.5 to 51.5 minutes


IQR outlier records: 270
Impossible nonpositive delivery times: 0
Treatment: retain valid IQR outliers and flag them in the report; do not delete them without evidence of error.


## 13. Categorical Data Validation

Inspect distinct labels after trimming whitespace. Normalization is limited to cosmetic leading/trailing whitespace, the standard `conditions ` weather prefix, and the known `Metropolitian` spelling variant. Other labels are not forced into invented categories.

In [12]:
CATEGORICAL_COLUMNS = [
    'Weatherconditions', 'Road_traffic_density', 'Type_of_vehicle',
    'Type_of_order', 'City',
]
category_labels = {}
for column in CATEGORICAL_COLUMNS:
    values = normalized_df[column]
    category_labels[column] = sorted(values.dropna().unique().tolist())
    print(f'{column} ({values.nunique(dropna=True)} non-missing labels): {category_labels[column]}')

print("Observed label correction candidate: 'Metropolitian' is standardized to 'Metropolitan'.")
print("Weather's 'conditions ' prefix is formatting and is removed; 'conditions NaN' was already treated as missing.")

Weatherconditions (6 non-missing labels): ['conditions Cloudy', 'conditions Fog', 'conditions Sandstorms', 'conditions Stormy', 'conditions Sunny', 'conditions Windy']
Road_traffic_density (4 non-missing labels): ['High', 'Jam', 'Low', 'Medium']
Type_of_vehicle (4 non-missing labels): ['bicycle', 'electric_scooter', 'motorcycle', 'scooter']
Type_of_order (4 non-missing labels): ['Buffet', 'Drinks', 'Meal', 'Snack']
City (3 non-missing labels): ['Metropolitian', 'Semi-Urban', 'Urban']
Observed label correction candidate: 'Metropolitian' is standardized to 'Metropolitan'.
Weather's 'conditions ' prefix is formatting and is removed; 'conditions NaN' was already treated as missing.


## 14. Data Cleaning and Treatment

Preserve rows with incomplete descriptive fields. Clean or invalidate only the affected values, and remove rows only when the delivery target is unusable or the full row is an exact duplicate. Maintain a per-issue audit table with affected and removed record counts.

In [13]:
audit_rows = []

def add_audit(issue, affected_records, treatment, records_removed=0):
    audit_rows.append({
        'issue': issue,
        'affected_records': int(affected_records),
        'treatment': treatment,
        'records_removed': int(records_removed),
    })

cleaned_df = normalized_df.copy()
cleaned_df['Delivery_person_Age_original'] = raw_df['Delivery_person_Age'].astype('string')
cleaned_df['Delivery_person_Age_invalid'] = age_invalid_mask.astype(bool)
cleaned_df['Delivery_person_Age_unparseable'] = age_unparseable_mask.astype(bool)
cleaned_df['Delivery_person_Age'] = age_values.mask(age_invalid_mask | age_unparseable_mask, pd.NA)
cleaned_df['Delivery_person_Ratings'] = rating_values.mask(rating_invalid_mask | rating_unparseable_mask, pd.NA)

# Retain invalid locations as missing coordinate pairs; never invent replacement GPS data.
for label, (lat_col, lon_col) in GPS_PAIRS.items():
    details = gps_details[label]
    invalid_pair = details['invalid']
    cleaned_df.loc[invalid_pair, [lat_col, lon_col]] = pd.NA
    cleaned_df[lat_col] = pd.to_numeric(cleaned_df[lat_col], errors='coerce')
    cleaned_df[lon_col] = pd.to_numeric(cleaned_df[lon_col], errors='coerce')

for column, parsed in parsed_datetimes.items():
    cleaned_df[column] = parsed.dt.strftime('%Y-%m-%d' if column == 'Order_Date' else '%H:%M:%S').astype('string')

for column in CATEGORICAL_COLUMNS:
    cleaned_df[column] = normalized_df[column]
cleaned_df['Weatherconditions'] = cleaned_df['Weatherconditions'].str.replace(r'^conditions\s+', '', regex=True, case=False)
cleaned_df['City'] = cleaned_df['City'].replace({'Metropolitian': 'Metropolitan'})

cleaned_df[TARGET_COLUMN] = target_minutes
duplicate_drop_mask = duplicate_mask
target_drop_mask = target_invalid_mask & ~duplicate_drop_mask
removed_mask = duplicate_drop_mask | target_invalid_mask
cleaned_df = cleaned_df.loc[~removed_mask].copy()
cleaned_df[TARGET_COLUMN] = cleaned_df[TARGET_COLUMN].astype('Int64') if cleaned_df[TARGET_COLUMN].dropna().mod(1).eq(0).all() else cleaned_df[TARGET_COLUMN]

add_audit('Missing values and sentinels', int(normalized_df.isna().any(axis=1).sum()), 'Standardize blank/sentinel values to missing; retain incomplete rows and do not impute.')
add_audit('Exact duplicate rows', duplicate_count, 'Drop repeated exact rows, keeping the first copy.', duplicate_count)
add_audit('Invalid rider ages', int((age_invalid_mask | age_unparseable_mask).sum()), 'Validate with inclusive 18-80 rule; set invalid/unparseable cleaned values missing, preserve raw age and row-level flags, and retain orders.')
add_audit('Invalid rider ratings', int((rating_invalid_mask | rating_unparseable_mask).sum()), 'Set values outside 1–5 or unparseable ratings to missing; retain orders.')
add_audit('Invalid/suspicious GPS records', int(gps_invalid_records.sum()), 'Set each affected coordinate pair to missing; retain orders and do not invent coordinates.')
add_audit('Malformed nonempty dates/times', int(pd.concat(date_time_issues, axis=1).any(axis=1).sum()), 'Set malformed date/time fields to missing; retain orders.')
add_audit('Missing order times', int(normalized_df['Time_Orderd'].isna().sum()), 'Keep as missing; do not infer an order time.')
add_audit('Invalid/missing/nonpositive delivery target', int(target_invalid_mask.sum()), 'Remove rows because delivery-time outcome is unusable.', int(target_invalid_mask.sum()))
add_audit('IQR delivery-time outliers', int(target_outlier_mask.sum()), 'Flag only; retain since an IQR flag alone does not establish a data error.')
category_changed = pd.Series(False, index=normalized_df.index)
for column in CATEGORICAL_COLUMNS:
    original_values = raw_df[column].astype('string')
    changed = original_values.ne(original_values.str.strip())
    if column == 'Weatherconditions':
        changed |= normalized_df[column].str.match(r'^conditions\s+', case=False, na=False)
    if column == 'City':
        changed |= normalized_df[column].eq('Metropolitian')
    category_changed |= changed.fillna(False)
add_audit('Categorical formatting/known spelling', int(category_changed.sum()), 'Trim labels, remove weather prefix, and standardize Metropolitian to Metropolitan; preserve other labels.')

quality_report = pd.DataFrame(audit_rows)
print(f'Rows before treatment: {len(raw_df):,}')
print(f'Rows retained: {len(cleaned_df):,}')
print(f'Rows removed (unique rows): {int(removed_mask.sum()):,}')
display(quality_report)

Rows before treatment: 45,593
Rows retained: 45,593
Rows removed (unique rows): 0


,issue,affected_records,treatment,records_removed
0,Missing values and sentinels,4225,Standardize blank/sentinel values to missing; retain incomplete rows and do not impute.,0
1,Exact duplicate rows,0,"Drop repeated exact rows, keeping the first copy.",0
2,Invalid rider ages,38,"Validate with inclusive 18-80 rule; set invalid/unparseable cleaned values missing, preserve raw age and row-level f...",0
3,Invalid rider ratings,53,Set values outside 1–5 or unparseable ratings to missing; retain orders.,0
4,Invalid/suspicious GPS records,3640,Set each affected coordinate pair to missing; retain orders and do not invent coordinates.,0
5,Malformed nonempty dates/times,0,Set malformed date/time fields to missing; retain orders.,0
6,Missing order times,1731,Keep as missing; do not infer an order time.,0
7,Invalid/missing/nonpositive delivery target,0,Remove rows because delivery-time outcome is unusable.,0
8,IQR delivery-time outliers,270,Flag only; retain since an IQR flag alone does not establish a data error.,0
9,Categorical formatting/known spelling,45593,"Trim labels, remove weather prefix, and standardize Metropolitian to Metropolitan; preserve other labels.",0


## 15. Before vs After Comparison

Compare row counts, missing values, duplicates, GPS flags, invalid rider measurements, and target outliers. A decrease in rows is explicitly attributable to the audit rules above.

In [14]:
missing_after = cleaned_df.isna().sum().rename('missing_after').to_frame()
missing_comparison = missing_before[['missing_count', 'missing_percent']].join(missing_after)
missing_comparison['missing_before'] = missing_comparison['missing_count']
missing_comparison = missing_comparison[['missing_before', 'missing_percent', 'missing_after']]
missing_comparison['missing_after_percent'] = (missing_comparison['missing_after'] / len(cleaned_df) * 100).round(2)
display(missing_comparison)

comparison = pd.DataFrame([
    {'measure': 'Rows', 'before': len(raw_df), 'after': len(cleaned_df)},
    {'measure': 'Missing cells (normalized)', 'before': int(normalized_df.isna().sum().sum()), 'after': int(cleaned_df.isna().sum().sum())},
    {'measure': 'Exact duplicate records', 'before': duplicate_count, 'after': int(cleaned_df.duplicated().sum())},
    {'measure': 'Invalid/suspicious GPS records detected', 'before': int(gps_invalid_records.sum()), 'after': 0},
    {'measure': 'Invalid rider ages detected', 'before': int((age_invalid_mask | age_unparseable_mask).sum()), 'after': 0},
    {'measure': 'Invalid rider ratings detected', 'before': int((rating_invalid_mask | rating_unparseable_mask).sum()), 'after': 0},
    {'measure': 'IQR delivery-time outliers retained', 'before': int(target_outlier_mask.sum()), 'after': int(cleaned_df[TARGET_COLUMN].between(lower_fence, upper_fence, inclusive='neither').eq(False).sum())},
])
display(comparison)
print('Age/rating/GPS “after” values mean invalid values have been set to missing; original issue counts remain in the audit report.')

,missing_before,missing_percent,missing_after,missing_after_percent
ID,0,0.00,0,0.00
Delivery_person_ID,0,0.00,0,0.00
Delivery_person_Age,1854,4.07,1892,4.15
Delivery_person_Ratings,1908,4.18,1961,4.30
Restaurant_latitude,0,0.00,3640,7.98
Restaurant_longitude,0,0.00,3640,7.98
Delivery_location_latitude,0,0.00,3640,7.98
Delivery_location_longitude,0,0.00,3640,7.98
Order_Date,0,0.00,0,0.00
Time_Orderd,1731,3.80,1731,3.80


,measure,before,after
0,Rows,45593,45593
1,Missing cells (normalized),9131,23782
2,Exact duplicate records,0,0
3,Invalid/suspicious GPS records detected,3640,0
4,Invalid rider ages detected,38,0
5,Invalid rider ratings detected,53,0
6,IQR delivery-time outliers retained,270,270


Age/rating/GPS “after” values mean invalid values have been set to missing; original issue counts remain in the audit report.


## 16. Final Data Quality Report

Summarize the observed issues in presentation-friendly language. The report distinguishes missing/invalid field values from removed records and documents why valid statistical outliers remain.

In [15]:
total_missing_before = int(normalized_df.isna().sum().sum())
total_missing_after = int(cleaned_df.isna().sum().sum())
retained_records = len(cleaned_df)
removed_records = int(removed_mask.sum())
invalid_age_count = int((age_invalid_mask | age_unparseable_mask).sum())
invalid_rating_count = int((rating_invalid_mask | rating_unparseable_mask).sum())
invalid_gps_count = int(gps_invalid_records.sum())
outlier_count = int(target_outlier_mask.sum())

final_observations = [
    f"The source contains {original_shape[0]:,} records and {original_shape[1]} columns.",
    f"Missing-value normalization found {total_missing_before:,} missing cells before treatment and {total_missing_after:,} in the retained cleaned data.",
    f"{duplicate_count:,} exact duplicate records were identified; {removed_records:,} records in total were removed under the duplicate/invalid-target rules.",
    f"{invalid_gps_count:,} records had invalid or suspicious GPS pairs. Coordinates were marked missing instead of guessed.",
    f"{invalid_age_count:,} rider ages failed the inclusive {AGE_MIN}-{AGE_MAX} validation rule or were unparseable; cleaned values were marked missing, original values and flags were retained, and orders were kept.",
    AGE_EXPLANATION,
    f"{outlier_count:,} delivery times fell outside the 1.5×IQR fences. They were retained and flagged, not automatically deleted.",
    "Other missing order attributes were preserved without imputation. Categorical formatting was standardized only where specified.",
]

print('FINAL DATA QUALITY SUMMARY')
print('=' * 72)
print(f'Original dataset shape: {original_shape}')
print(f'Cleaned dataset shape: {cleaned_df.shape}')
print(f'Missing cells before / after: {total_missing_before:,} / {total_missing_after:,}')
print(f'Exact duplicate records: {duplicate_count:,}')
print(f'Invalid/suspicious GPS records: {invalid_gps_count:,}')
print(f'Invalid rider ages / ratings: {invalid_age_count:,} / {invalid_rating_count:,}')
print(f'Delivery-time IQR outliers (retained): {outlier_count:,}')
print(f'Records retained / removed: {retained_records:,} / {removed_records:,}')
print('\nSimple-English observations:')
for observation in final_observations:
    print(f'- {observation}')

display(quality_report)

FINAL DATA QUALITY SUMMARY
Original dataset shape: (45593, 20)
Cleaned dataset shape: (45593, 23)
Missing cells before / after: 9,131 / 23,782
Exact duplicate records: 0
Invalid/suspicious GPS records: 3,640
Invalid rider ages / ratings: 38 / 53
Delivery-time IQR outliers (retained): 270
Records retained / removed: 45,593 / 0

Simple-English observations:
- The source contains 45,593 records and 20 columns.
- Missing-value normalization found 9,131 missing cells before treatment and 23,782 in the retained cleaned data.
- 0 exact duplicate records were identified; 0 records in total were removed under the duplicate/invalid-target rules.
- 3,640 records had invalid or suspicious GPS pairs. Coordinates were marked missing instead of guessed.
- 38 rider ages failed the inclusive 18-80 validation rule or were unparseable; cleaned values were marked missing, original values and flags were retained, and orders were kept.
- For consistency across the project, rider age values are considered va

,issue,affected_records,treatment,records_removed
0,Missing values and sentinels,4225,Standardize blank/sentinel values to missing; retain incomplete rows and do not impute.,0
1,Exact duplicate rows,0,"Drop repeated exact rows, keeping the first copy.",0
2,Invalid rider ages,38,"Validate with inclusive 18-80 rule; set invalid/unparseable cleaned values missing, preserve raw age and row-level f...",0
3,Invalid rider ratings,53,Set values outside 1–5 or unparseable ratings to missing; retain orders.,0
4,Invalid/suspicious GPS records,3640,Set each affected coordinate pair to missing; retain orders and do not invent coordinates.,0
5,Malformed nonempty dates/times,0,Set malformed date/time fields to missing; retain orders.,0
6,Missing order times,1731,Keep as missing; do not infer an order time.,0
7,Invalid/missing/nonpositive delivery target,0,Remove rows because delivery-time outcome is unusable.,0
8,IQR delivery-time outliers,270,Flag only; retain since an IQR flag alone does not establish a data error.,0
9,Categorical formatting/known spelling,45593,"Trim labels, remove weather prefix, and standardize Metropolitian to Metropolitan; preserve other labels.",0


## 17. Save Cleaned Dataset

Create the outputs directory automatically and save the cleaned table, issue-level CSV report, and readable summary. Exceptions are reported with their path and are not silently ignored.

In [16]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
try:
    cleaned_df.to_csv(CLEANED_PATH, index=False, na_rep='')
    quality_report.to_csv(REPORT_PATH, index=False)
    summary_lines = [
        'Quick-Commerce Delivery Intelligence — Problem Statement 1: Data Quality & Reliability',
        '',
        f'Original dataset shape: {original_shape}',
        f'Cleaned dataset shape: {cleaned_df.shape}',
        f'Missing cells before / after: {total_missing_before:,} / {total_missing_after:,}',
        f'Exact duplicate records: {duplicate_count:,}',
        f'Invalid/suspicious GPS records: {invalid_gps_count:,}',
        f'Invalid rider ages: {invalid_age_count:,}',
        f'Invalid rider ratings: {invalid_rating_count:,}',
        f'Delivery-time IQR outliers (retained): {outlier_count:,}',
        f'Records retained: {retained_records:,}',
        f'Records removed: {removed_records:,}',
        '',
        'Final observations:',
        *[f'- {item}' for item in final_observations],
        '',
        'Treatment note: no missing values were imputed and no replacement GPS coordinates were invented.',
        'IQR outliers were retained; only exact duplicates and records with unusable delivery-time targets were removed.',
    ]
    SUMMARY_PATH.write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
except OSError as exc:
    raise RuntimeError(f'Unable to save one or more data-quality outputs: {exc}') from exc

print('Saved outputs:')
for output_path in [CLEANED_PATH, REPORT_PATH, SUMMARY_PATH]:
    print(f'- {output_path.relative_to(PROJECT_ROOT)} ({output_path.stat().st_size:,} bytes)')

Saved outputs:
- data\cleaned_data.csv (7,260,927 bytes)
- outputs\data_quality_report.csv (1,189 bytes)
- outputs\data_quality_summary.txt (1,702 bytes)


## 18. Missing Value Analysis

The earlier quality checks converted invalid ages, ratings, and placeholder GPS pairs to missing values. This section separates those validation-created missing values from fields that were already absent in the source. It reports missing counts and percentages for every original column, affected rows, and the most common missingness patterns before any new imputation.

The original sections above are the first-pass audit. Their counts are the pre-treatment baseline; the final treatment results and deliverables are reported in Sections 18?24 below.

In [17]:
# Snapshot the current pre-imputation data; this is the missing-value audit baseline.
missing_before_treatment = cleaned_df.isna().sum().rename('missing_count').to_frame()
missing_before_treatment['missing_percent'] = (missing_before_treatment['missing_count'] / len(cleaned_df) * 100).round(2)
missing_before_treatment['rows_affected'] = cleaned_df.isna().sum().astype(int)
display(missing_before_treatment)

missing_pattern_labels = cleaned_df.isna().apply(
    lambda row: ', '.join(row.index[row].tolist()) if row.any() else 'No missing values', axis=1
)
missing_patterns = (
    missing_pattern_labels.value_counts()
    .rename_axis('missing_columns_pattern')
    .reset_index(name='record_count')
)
missing_patterns['record_percent'] = (missing_patterns['record_count'] / len(cleaned_df) * 100).round(2)
print(f'Unique missingness patterns: {len(missing_patterns):,}; showing the 20 most common.')
display(missing_patterns.head(20))

,missing_count,missing_percent,rows_affected
ID,0,0.00,0
Delivery_person_ID,0,0.00,0
Delivery_person_Age,1892,4.15,1892
Delivery_person_Ratings,1961,4.30,1961
Restaurant_latitude,3640,7.98,3640
Restaurant_longitude,3640,7.98,3640
Delivery_location_latitude,3640,7.98,3640
Delivery_location_longitude,3640,7.98,3640
Order_Date,0,0.00,0
Time_Orderd,1731,3.80,1731


Unique missingness patterns: 47; showing the 20 most common.


,missing_columns_pattern,record_count,record_percent
0,No missing values,38064,83.49
1,"Restaurant_latitude, Restaurant_longitude, Delivery_location_latitude, Delivery_location_longitude",3304,7.25
2,City,1004,2.20
3,"Delivery_person_Age, Delivery_person_Ratings, Time_Orderd",956,2.10
4,multiple_deliveries,808,1.77
5,"Delivery_person_Age, Delivery_person_Ratings, Time_Orderd, Weatherconditions, Road_traffic_density",442,0.97
6,"Delivery_person_Age, Delivery_person_Ratings",184,0.40
7,Festival,179,0.39
8,"Restaurant_latitude, Restaurant_longitude, Delivery_location_latitude, Delivery_location_longitude, multiple_deliveries",84,0.18
9,"Delivery_person_Age, Delivery_person_Ratings, Restaurant_latitude, Restaurant_longitude, Delivery_location_latitude,...",83,0.18


## 19. Missing Value Treatment Strategy

The input to this treatment is the existing cleaned table: it retains orders and has already marked invalid age, rating, and placeholder GPS values as missing. We compare those values with the normalized original source to identify whether each missing value was originally absent or was created by a validation rule. Invalid-coordinate flags and timestamp missing indicators will be kept. Nothing in this step removes rows.

In [18]:
# Count cells originally absent in the source separately from invalid values marked missing by validation.
source_missing_counts = normalized_df.isna().sum().reindex(cleaned_df.columns, fill_value=0).astype(int)
invalid_to_missing_counts = pd.Series(0, index=cleaned_df.columns, dtype='int64')
invalid_to_missing_counts['Delivery_person_Age'] = int((age_invalid_mask | age_unparseable_mask).sum())
invalid_to_missing_counts['Delivery_person_Age_original'] = 0
invalid_to_missing_counts['Delivery_person_Age_invalid'] = 0
invalid_to_missing_counts['Delivery_person_Age_unparseable'] = 0
invalid_to_missing_counts['Delivery_person_Ratings'] = int((rating_invalid_mask | rating_unparseable_mask).sum())
for gps_name, (latitude_column, longitude_column) in GPS_PAIRS.items():
    gps_invalid_mask = gps_details[gps_name]['invalid']
    invalid_to_missing_counts[latitude_column] = int(gps_invalid_mask.sum())
    invalid_to_missing_counts[longitude_column] = int(gps_invalid_mask.sum())
for datetime_column, malformed_mask in date_time_issues.items():
    invalid_to_missing_counts[datetime_column] = int(malformed_mask.sum())

missing_provenance = pd.DataFrame({
    'originally_absent_cells': source_missing_counts,
    'invalid_values_converted_to_missing': invalid_to_missing_counts,
    'missing_before_treatment': missing_before_treatment['missing_count'],
})
missing_provenance['unattributed_difference'] = (
    missing_provenance['missing_before_treatment']
    - missing_provenance['originally_absent_cells']
    - missing_provenance['invalid_values_converted_to_missing']
).clip(lower=0)
print('Missing-value provenance, by original column:')
display(missing_provenance)
print('Treatment policy: median for age, rating, and delivery-count inputs; Unknown for selected categories;')
print('preserve missing timestamps and GPS. Do not impute the delivery-time target.')

Missing-value provenance, by original column:


,originally_absent_cells,invalid_values_converted_to_missing,missing_before_treatment,unattributed_difference
ID,0,0,0,0
Delivery_person_ID,0,0,0,0
Delivery_person_Age,1854,38,1892,0
Delivery_person_Ratings,1908,53,1961,0
Restaurant_latitude,0,3640,3640,0
Restaurant_longitude,0,3640,3640,0
Delivery_location_latitude,0,3640,3640,0
Delivery_location_longitude,0,3640,3640,0
Order_Date,0,0,0,0
Time_Orderd,1731,0,1731,0


Treatment policy: median for age, rating, and delivery-count inputs; Unknown for selected categories;
preserve missing timestamps and GPS. Do not impute the delivery-time target.


## 20. Column-wise Imputation

Use the median for `Delivery_person_Age`, `Delivery_person_Ratings`, and `multiple_deliveries`; use the literal `Unknown` category for weather, traffic, festival, and city. For age, the validity rule is inclusive 18?80. Original source age text and row-level invalid/unparseable flags are retained in the cleaned output alongside the median-treated age field. Medians in this **Problem Statement 1 export** are calculated from the available valid observations so judges can inspect a complete descriptive file. This full-dataset export is not a training matrix: for any later model, start from the pre-imputation cleaned data and fit the pipeline only on the training partition, as demonstrated below.

In [19]:
from sklearn.impute import SimpleImputer

final_cleaned_df = cleaned_df.copy()
numeric_impute_columns = ['Delivery_person_Age', 'Delivery_person_Ratings', 'multiple_deliveries']
categorical_impute_columns = ['Weatherconditions', 'Road_traffic_density', 'Festival', 'City']

numeric_missing_counts = final_cleaned_df[numeric_impute_columns].isna().sum().astype(int)
median_imputer = SimpleImputer(strategy='median')
final_cleaned_df[numeric_impute_columns] = median_imputer.fit_transform(final_cleaned_df[numeric_impute_columns])

categorical_missing_counts = final_cleaned_df[categorical_impute_columns].isna().sum().astype(int)
for column in categorical_impute_columns:
    final_cleaned_df[column] = final_cleaned_df[column].fillna('Unknown')

imputation_audit = pd.DataFrame({
    'column': numeric_impute_columns + categorical_impute_columns,
    'missing_values_filled': [int(numeric_missing_counts[c]) for c in numeric_impute_columns]
        + [int(categorical_missing_counts[c]) for c in categorical_impute_columns],
    'treatment': ['Median imputation'] * len(numeric_impute_columns)
        + ["Fill missing category with 'Unknown'"] * len(categorical_impute_columns),
    'fitted_median_or_value': [float(value) for value in median_imputer.statistics_]
        + ['Unknown'] * len(categorical_impute_columns),
})
print('Applied column-wise treatment (counts are missing values filled, not rows deleted):')
display(imputation_audit)
print('No records were removed during imputation.')

Applied column-wise treatment (counts are missing values filled, not rows deleted):


,column,missing_values_filled,treatment,fitted_median_or_value
0,Delivery_person_Age,1892,Median imputation,30.0
1,Delivery_person_Ratings,1961,Median imputation,4.7
2,multiple_deliveries,993,Median imputation,1.0
3,Weatherconditions,616,Fill missing category with 'Unknown',Unknown
4,Road_traffic_density,601,Fill missing category with 'Unknown',Unknown
5,Festival,228,Fill missing category with 'Unknown',Unknown
6,City,1200,Fill missing category with 'Unknown',Unknown


No records were removed during imputation.


## 21. GPS and Timestamp Handling

Missing or invalid GPS values are never replaced. Preserve row-level flags so downstream geographic work can exclude these orders while non-geographic analyses can still use their other fields. Preserve timestamp gaps and add indicators rather than inventing times. `Time_Order_picked` is after order placement; neither it nor its missingness indicator should be an order-placement feature.

In [20]:
# Preserve validation outcomes and make geographic eligibility explicit without deleting records.
final_cleaned_df['restaurant_gps_invalid'] = gps_details['restaurant']['invalid'].astype(bool)
final_cleaned_df['delivery_gps_invalid'] = gps_details['delivery']['invalid'].astype(bool)
final_cleaned_df['gps_invalid_any'] = final_cleaned_df['restaurant_gps_invalid'] | final_cleaned_df['delivery_gps_invalid']
final_cleaned_df['gps_geographic_eligible'] = ~final_cleaned_df['gps_invalid_any']

# Missing indicators preserve whether timestamps were unavailable; values themselves stay missing.
timestamp_columns = ['Order_Date', 'Time_Orderd', 'Time_Order_picked']
for column in timestamp_columns:
    final_cleaned_df[f'{column}_missing'] = cleaned_df[column].isna().astype(bool)

gps_preserved_count = int(final_cleaned_df['gps_invalid_any'].sum())
gps_geographic_excluded_count = int((~final_cleaned_df['gps_geographic_eligible']).sum())
print('Invalid GPS records preserved in the dataset:', f'{gps_preserved_count:,}')
print('Records to exclude from geographic distance/clustering work:', f'{gps_geographic_excluded_count:,}')
print('Timestamp missingness (value remains missing; indicator added):')
display(pd.DataFrame({
    'missing_timestamp_values': final_cleaned_df[timestamp_columns].isna().sum(),
    'missing_indicators_true': [int(final_cleaned_df[f'{c}_missing'].sum()) for c in timestamp_columns],
}))
print('The pickup-time field occurs after order placement and must not be used for order-placement predictions.')

Invalid GPS records preserved in the dataset: 3,640
Records to exclude from geographic distance/clustering work: 3,640
Timestamp missingness (value remains missing; indicator added):


,missing_timestamp_values,missing_indicators_true
Order_Date,0,0
Time_Orderd,1731,1731
Time_Order_picked,0,0


The pickup-time field occurs after order placement and must not be used for order-placement predictions.


## 22. Before vs After Comparison

The comparison is at the original 20-column level, so the extra audit indicators do not disguise missing source values. GPS and timestamps are expected to remain missing. Missing target values are never imputed; if present, they are excluded only from supervised training, not silently deleted from this quality export.

In [21]:
missing_after_treatment = final_cleaned_df[cleaned_df.columns].isna().sum().rename('missing_after')
missing_comparison = missing_before_treatment[['missing_count', 'missing_percent']].join(missing_after_treatment)
missing_comparison = missing_comparison.rename(columns={
    'missing_count': 'missing_before', 'missing_percent': 'missing_before_percent'
})
missing_comparison['missing_after_percent'] = (missing_comparison['missing_after'] / len(final_cleaned_df) * 100).round(2)
print('Per-column missing values before and after treatment:')
display(missing_comparison)

missing_target_count = int(final_cleaned_df[TARGET_COLUMN].isna().sum())
supervised_excluded_for_target = missing_target_count
records_retained = len(final_cleaned_df)
records_excluded_from_export = len(cleaned_df) - records_retained
comparison_summary = pd.DataFrame([
    {'measure': 'Records in pre-treatment data', 'count': len(cleaned_df)},
    {'measure': 'Records retained in final export', 'count': records_retained},
    {'measure': 'Records excluded from final export', 'count': records_excluded_from_export},
    {'measure': 'Invalid GPS records preserved', 'count': gps_preserved_count},
    {'measure': 'Invalid GPS records excluded from geographic analysis', 'count': gps_geographic_excluded_count},
    {'measure': 'Missing delivery-time targets', 'count': missing_target_count},
    {'measure': 'Rows excluded from supervised training due to missing target', 'count': supervised_excluded_for_target},
])
display(comparison_summary)
print(f"Total missing cells before: {int(missing_before_treatment['missing_count'].sum()):,}")
print(f"Total missing cells after: {int(final_cleaned_df[cleaned_df.columns].isna().sum().sum()):,}")

Per-column missing values before and after treatment:


,missing_before,missing_before_percent,missing_after,missing_after_percent
ID,0,0.00,0,0.00
Delivery_person_ID,0,0.00,0,0.00
Delivery_person_Age,1892,4.15,0,0.00
Delivery_person_Ratings,1961,4.30,0,0.00
Restaurant_latitude,3640,7.98,3640,7.98
Restaurant_longitude,3640,7.98,3640,7.98
Delivery_location_latitude,3640,7.98,3640,7.98
Delivery_location_longitude,3640,7.98,3640,7.98
Order_Date,0,0.00,0,0.00
Time_Orderd,1731,3.80,1731,3.80


,measure,count
0,Records in pre-treatment data,45593
1,Records retained in final export,45593
2,Records excluded from final export,0
3,Invalid GPS records preserved,3640
4,Invalid GPS records excluded from geographic analysis,3640
5,Missing delivery-time targets,0
6,Rows excluded from supervised training due to missing target,0


Total missing cells before: 23,782
Total missing cells after: 16,291


## 23. Final Data Validation

Validate that treatments changed values only under the documented rules and did not lose orders. The feature list below is an illustrative PS1 preprocessing demonstration, not the authority for order-placement modelling; Module 02's feature availability report is the single source of truth for future machine-learning inputs. The preprocessing example below does not train a model: it demonstrates how to split first, fit imputers/encoders on training features only, and then transform validation features. The target, rider rating, pickup timestamp, delivery count, and GPS columns are excluded from this illustrative feature set. Rider age is included only for a dispatch-time use case where the assigned rider is already known; exclude it for a prediction made at order placement. For consistency across the project, rider age values are considered valid for analysis when they fall between 18 and 80 years, inclusive. This is a data-validation rule and does not represent rider professional experience.

In [22]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# Leakage-safe preprocessing illustration only; no estimator/model is trained here.
# Module 02's feature availability report is authoritative for order-placement models.
# These features are available at/around order creation except rider age, which requires
# a dispatch-time prediction after assignment. Never use pickup time or delivery target.
training_numeric_features = ['Delivery_person_Age']
training_categorical_features = [
    'Weatherconditions', 'Road_traffic_density', 'Type_of_order', 'Festival', 'City'
]
training_feature_columns = training_numeric_features + training_categorical_features

numeric_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])
categorical_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='Unknown')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_preprocessor, training_numeric_features),
    ('categorical', categorical_preprocessor, training_categorical_features),
])

supervised_rows = cleaned_df.loc[cleaned_df[TARGET_COLUMN].notna()]
X = supervised_rows[training_feature_columns].astype(object).where(
    supervised_rows[training_feature_columns].notna(), np.nan
)
y = supervised_rows[TARGET_COLUMN]
X_train, X_validation, y_train, y_validation = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train_transformed = preprocessor.fit_transform(X_train)
X_validation_transformed = preprocessor.transform(X_validation)
print('Preprocessor demonstration only (no model fitted):')
print(f'Train rows: {len(X_train):,}; validation rows: {len(X_validation):,}')
print('Transformed train / validation shapes:', X_train_transformed.shape, X_validation_transformed.shape)
print('The imputer was fit on X_train only; validation data was transformed with those fitted values.')
print('For order-placement predictions, remove Delivery_person_Age unless it is known at that exact prediction time.')

# Validate exported cleaning guarantees.
assert AGE_MIN == 18 and AGE_MAX == 80, 'Age validation must use inclusive 18-80 range.'
assert final_cleaned_df['Delivery_person_Age_original'].equals(cleaned_df['Delivery_person_Age_original'])
assert final_cleaned_df['Delivery_person_Age_invalid'].equals(cleaned_df['Delivery_person_Age_invalid'])
assert final_cleaned_df['Delivery_person_Age_unparseable'].equals(cleaned_df['Delivery_person_Age_unparseable'])
assert not pd.to_numeric(final_cleaned_df.loc[final_cleaned_df['Delivery_person_Age_invalid'], 'Delivery_person_Age_original'].astype('string').str.strip(), errors='coerce').between(AGE_MIN, AGE_MAX, inclusive='both').any()
assert len(final_cleaned_df) == len(cleaned_df), 'The treatment step must not remove records.'
assert final_cleaned_df[numeric_impute_columns].notna().all().all(), 'Selected numeric inputs should be imputed.'
assert final_cleaned_df[categorical_impute_columns].notna().all().all(), 'Selected categorical inputs should be filled.'
assert final_cleaned_df[TARGET_COLUMN].isna().sum() == cleaned_df[TARGET_COLUMN].isna().sum(), 'Target missingness must not change.'
assert final_cleaned_df.loc[final_cleaned_df['restaurant_gps_invalid'], ['Restaurant_latitude', 'Restaurant_longitude']].isna().all().all(), 'Invalid restaurant coordinates must remain missing.'
assert final_cleaned_df.loc[final_cleaned_df['delivery_gps_invalid'], ['Delivery_location_latitude', 'Delivery_location_longitude']].isna().all().all(), 'Invalid delivery coordinates must remain missing.'
assert final_cleaned_df['gps_invalid_any'].sum() == gps_preserved_count
assert final_cleaned_df[timestamp_columns].isna().sum().equals(cleaned_df[timestamp_columns].isna().sum())
print('All final data-quality assertions passed.')

Preprocessor demonstration only (no model fitted):
Train rows: 36,474; validation rows: 9,119
Transformed train / validation shapes: (36474, 24) (9119, 24)
The imputer was fit on X_train only; validation data was transformed with those fitted values.
For order-placement predictions, remove Delivery_person_Age unless it is known at that exact prediction time.
All final data-quality assertions passed.


## 24. Final Problem Statement 1 Summary

This final section reports the treatment outcome in straightforward terms and saves the requested deliverables. Missing GPS and timestamps are expected to remain missing; that is safer than fabricating a location or time. The pre-imputation `cleaned_data.csv` remains the appropriate starting point for future modelling pipelines.

In [23]:
# Build a single per-column audit, including provenance, treatment, and remaining missingness.
treatment_by_column = {column: 'Median imputation (full-data descriptive export; refit train-only for ML)' for column in numeric_impute_columns}
treatment_by_column.update({column: "Fill with 'Unknown'" for column in categorical_impute_columns})
treatment_by_column.update({column: 'Preserve missing value; add missing indicator' for column in timestamp_columns})
for column in GPS_PAIRS.values():
    for coordinate_column in column:
        treatment_by_column[coordinate_column] = 'Never impute; preserve missing coordinate and GPS invalid flag'
treatment_by_column[TARGET_COLUMN] = 'Never impute; exclude missing-target rows from supervised training only'

treatment_report = missing_provenance.copy()
treatment_report['treatment'] = [treatment_by_column.get(column, 'Preserve; no imputation applied') for column in treatment_report.index]
treatment_report['values_filled'] = 0
for column in numeric_impute_columns:
    treatment_report.loc[column, 'values_filled'] = int(numeric_missing_counts[column])
for column in categorical_impute_columns:
    treatment_report.loc[column, 'values_filled'] = int(categorical_missing_counts[column])
treatment_report['missing_after_treatment'] = final_cleaned_df[cleaned_df.columns].isna().sum()
treatment_report = treatment_report.reset_index(names='column')

numeric_medians_summary = ', '.join(
    f'{column}={median_imputer.statistics_[index]:g}'
    for index, column in enumerate(numeric_impute_columns)
)
total_original_missing_cells = int(source_missing_counts.sum())
total_validation_invalid_cells = int(invalid_to_missing_counts.sum())

final_observations = [
    f"The source contains {original_shape[0]:,} records and {original_shape[1]} original columns; {records_retained:,} records are retained in the final export.",
    f"{int(missing_before_treatment['missing_count'].sum()):,} missing cells existed before treatment ({total_original_missing_cells:,} originally absent; {total_validation_invalid_cells:,} invalidated by validation); {int(final_cleaned_df[cleaned_df.columns].isna().sum().sum()):,} remain afterward.",
    f"Age values are validated against the inclusive {AGE_MIN}-{AGE_MAX} rule; {invalid_age_count:,} invalid/unparseable ages are audited, their raw values and flags are preserved, and median imputation is used only in the descriptive export.",
    AGE_EXPLANATION,
    f"Rider rating and multiple-delivery count use their observed medians; selected categorical fields use Unknown.",
    f"{gps_preserved_count:,} records with invalid/suspicious GPS remain available for non-geographic work and are flagged out of geographic analysis.",
    f"Timestamp gaps are preserved with missing indicators. No timestamp or coordinate is invented.",
    f"{missing_target_count:,} delivery-time targets are missing; these are not imputed and would be excluded from supervised training.",
    "For future machine learning, split first and fit preprocessing only on training data; the displayed pipeline demonstrates that practice without training a model.",
]

print('FINAL PROBLEM STATEMENT 1 SUMMARY')
print('=' * 72)
print(f'Pre-treatment rows: {len(cleaned_df):,}; final rows: {len(final_cleaned_df):,}; excluded from export: {records_excluded_from_export:,}')
print(f'Missing cells before / after: {int(missing_before_treatment["missing_count"].sum()):,} / {int(final_cleaned_df[cleaned_df.columns].isna().sum().sum()):,}')
print(f'Invalid GPS preserved / excluded from geographic use: {gps_preserved_count:,} / {gps_geographic_excluded_count:,}')
print(f'Missing targets / supervised rows excluded: {missing_target_count:,} / {supervised_excluded_for_target:,}')
for item in final_observations:
    print(f'- {item}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_CLEANED_PATH = PROJECT_ROOT / 'data' / 'final_cleaned_data.csv'
TREATMENT_REPORT_PATH = OUTPUT_DIR / 'missing_value_treatment_report.csv'
FINAL_SUMMARY_PATH = OUTPUT_DIR / 'data_quality_final_summary.txt'
try:
    final_cleaned_df.to_csv(FINAL_CLEANED_PATH, index=False, na_rep='')
    treatment_report.to_csv(TREATMENT_REPORT_PATH, index=False)
    summary_lines = [
        'Quick-Commerce Delivery Intelligence ? Problem Statement 1: Missing Value Treatment',
        '',
        f'Original dataset shape: {original_shape}',
        f'Pre-treatment rows: {len(cleaned_df):,}',
        f'Final cleaned dataset shape: {final_cleaned_df.shape}',
        f'Records retained / excluded from final export: {records_retained:,} / {records_excluded_from_export:,}',
        f'Missing cells before / after treatment: {int(missing_before_treatment["missing_count"].sum()):,} / {int(final_cleaned_df[cleaned_df.columns].isna().sum().sum()):,}',
        f'Originally absent / invalidated by validation: {total_original_missing_cells:,} / {total_validation_invalid_cells:,}',
        f'Median values used: {numeric_medians_summary}',
        f'Invalid GPS records preserved / excluded from geographic analysis: {gps_preserved_count:,} / {gps_geographic_excluded_count:,}',
        f'Missing delivery targets / excluded from supervised training: {missing_target_count:,} / {supervised_excluded_for_target:,}',
        '',
        'Final observations:',
        *[f'- {item}' for item in final_observations],
        '',
        'Important: final_cleaned_data.csv uses dataset-wide medians for this descriptive cleaning deliverable. Do not use it as a pre-split model matrix. For machine learning, start from data/cleaned_data.csv, split first, then fit a Pipeline/ColumnTransformer only on training data.',
        'Invalid GPS pairs and missing timestamps remain missing. No coordinate or timestamp was invented. No rows were removed in the missing-value treatment step.',
    ]
    FINAL_SUMMARY_PATH.write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
except OSError as exc:
    raise RuntimeError(f'Could not write final missing-value treatment outputs: {exc}') from exc

print('Saved final deliverables:')
for output_path in [FINAL_CLEANED_PATH, TREATMENT_REPORT_PATH, FINAL_SUMMARY_PATH]:
    print(f'- {output_path.relative_to(PROJECT_ROOT)} ({output_path.stat().st_size:,} bytes)')
display(treatment_report)

FINAL PROBLEM STATEMENT 1 SUMMARY
Pre-treatment rows: 45,593; final rows: 45,593; excluded from export: 0
Missing cells before / after: 23,782 / 16,291
Invalid GPS preserved / excluded from geographic use: 3,640 / 3,640
Missing targets / supervised rows excluded: 0 / 0
- The source contains 45,593 records and 20 original columns; 45,593 records are retained in the final export.
- 23,782 missing cells existed before treatment (9,131 originally absent; 14,651 invalidated by validation); 16,291 remain afterward.
- Age values are validated against the inclusive 18-80 rule; 38 invalid/unparseable ages are audited, their raw values and flags are preserved, and median imputation is used only in the descriptive export.
- For consistency across the project, rider age values are considered valid for analysis when they fall between 18 and 80 years, inclusive. This is a data-validation rule and does not represent rider professional experience.
- Rider rating and multiple-delivery count use their o

Saved final deliverables:
- data\final_cleaned_data.csv (9,332,925 bytes)
- outputs\missing_value_treatment_report.csv (2,011 bytes)
- outputs\data_quality_final_summary.txt (2,374 bytes)


,column,originally_absent_cells,invalid_values_converted_to_missing,missing_before_treatment,unattributed_difference,treatment,values_filled,missing_after_treatment
0,ID,0,0,0,0,Preserve; no imputation applied,0,0
1,Delivery_person_ID,0,0,0,0,Preserve; no imputation applied,0,0
2,Delivery_person_Age,1854,38,1892,0,Median imputation (full-data descriptive export; refit train-only for ML),1892,0
3,Delivery_person_Ratings,1908,53,1961,0,Median imputation (full-data descriptive export; refit train-only for ML),1961,0
4,Restaurant_latitude,0,3640,3640,0,Never impute; preserve missing coordinate and GPS invalid flag,0,3640
5,Restaurant_longitude,0,3640,3640,0,Never impute; preserve missing coordinate and GPS invalid flag,0,3640
6,Delivery_location_latitude,0,3640,3640,0,Never impute; preserve missing coordinate and GPS invalid flag,0,3640
7,Delivery_location_longitude,0,3640,3640,0,Never impute; preserve missing coordinate and GPS invalid flag,0,3640
8,Order_Date,0,0,0,0,Preserve missing value; add missing indicator,0,0
9,Time_Orderd,1731,0,1731,0,Preserve missing value; add missing indicator,0,1731


## PS1 – Final GPS and Vehicle Data Validation

This final audit validates the previously cleaned dataset without restarting the cleaning process. The local project has no data dictionary. A published data dictionary for the same Kaggle source/schema defines `Vehicle_condition` values as 0, 1, and 2 ([data dictionary](https://github.com/Chandansahu18/food-delivery-operations-analytics-dashboard/blob/42683f1123fd417ceec94e5a749aee0293ee42c0/docs/data_dictionary.md); it cites the Kaggle Zomato Delivery Operations dataset). Therefore, observed value `3` is outside the documented domain: keep it in `Vehicle_condition_original`, mark the row, and set only the validated `Vehicle_condition` value to missing. No order is deleted.

For India plausibility, use a deliberately broad screening box of 6.5 to 37.5 degrees N and 68 to 98 degrees E. This is not a city geofence; it includes India's mainland and island extent with margin. Coordinates are also checked for missingness, zeros, placeholders, and global latitude/longitude limits. Negative values are not rejected merely because of their sign: the dataset's 431 negative restaurant latitudes are flagged because they fall outside India's bounds while the paired customer locations are in India. Invalid coordinates remain as recorded (or missing if the earlier cleaning already marked them missing), are never imputed, and are excluded from geographic calculations.

Distance is computed with the Haversine formula only when both restaurant and delivery coordinate pairs pass these checks. Large-distance investigation flags use the upper 1.5 times the IQR fence; flagged distances are retained, not deleted.

In [24]:
# This section is intentionally an audit on the already-cleaned export.
# It does not repeat missing-value treatments or remove any orders.
from pathlib import Path
import numpy as np
import pandas as pd

ps1_input_path = PROJECT_ROOT / 'data' / 'final_cleaned_data.csv'
ps1_source_path = PROJECT_ROOT / 'data' / 'train.csv'
ps1_output_dir = PROJECT_ROOT / 'outputs'
ps1_vehicle_report_path = ps1_output_dir / 'vehicle_condition_report.csv'
ps1_gps_report_path = ps1_output_dir / 'gps_validation_report.csv'
ps1_summary_path = ps1_output_dir / 'ps1_final_quality_summary.txt'

try:
    ps1_before = pd.read_csv(ps1_input_path, keep_default_na=True)
    ps1_source = pd.read_csv(ps1_source_path, dtype='string', keep_default_na=False)
except (OSError, pd.errors.ParserError, UnicodeError) as exc:
    raise RuntimeError(f'Could not read the existing final dataset or its source for audit: {exc}') from exc

ps1_records_before = len(ps1_before)
ps1_original_columns = ps1_before.columns.tolist()
ps1_target_before = pd.to_numeric(ps1_before[TARGET_COLUMN], errors='coerce').copy()
ps1_full_duplicates_before = int(ps1_before.duplicated().sum())
ps1_ids_before = ps1_before['ID'].astype('string').str.strip()
if ps1_ids_before.duplicated().any():
    raise ValueError('Order ID values are not unique in the existing final dataset; investigate before continuing.')

# Reconcile target values to the untouched source by order ID before adding audit columns.
source_audit = ps1_source.copy()
source_audit['__order_key'] = source_audit['ID'].astype('string').str.strip()
source_target_text = source_audit[TARGET_COLUMN].astype('string').str.strip()
source_target_numeric = pd.to_numeric(
    source_target_text.str.extract(r'^\(min\)\s*([+-]?\d+(?:\.\d+)?)\s*$', expand=False),
    errors='coerce',
)
source_audit['__source_target_numeric'] = source_target_numeric
source_audit['__age_original'] = source_audit['Delivery_person_Age'].astype('string')
if source_audit['__order_key'].duplicated().any():
    raise ValueError('Order IDs are not unique in data/train.csv; cannot verify target preservation one-to-one.')
source_target_by_id = source_audit.set_index('__order_key')['__source_target_numeric']
source_age_by_id = source_audit.set_index('__order_key')['__age_original']
final_target_by_id = pd.Series(ps1_target_before.to_numpy(), index=ps1_ids_before.to_numpy())
aligned_source_target = source_target_by_id.reindex(final_target_by_id.index)
if aligned_source_target.isna().any() or not np.array_equal(
    aligned_source_target.to_numpy(dtype=float), final_target_by_id.to_numpy(dtype=float)
):
    raise AssertionError('The current delivery-time target does not match the source values by order ID.')

# Preserve the original vehicle values, even when the validated field is set missing.
if 'Vehicle_condition_original' in ps1_before.columns:
    vehicle_original = pd.to_numeric(ps1_before['Vehicle_condition_original'], errors='coerce')
else:
    vehicle_original = pd.to_numeric(ps1_before['Vehicle_condition'], errors='coerce')
vehicle_valid_values = {0, 1, 2}
vehicle_invalid = vehicle_original.isna() | ~vehicle_original.isin(vehicle_valid_values)

ps1_result = ps1_before.copy()
aligned_source_age = source_age_by_id.reindex(ps1_ids_before.to_numpy())
if aligned_source_age.isna().any():
    raise AssertionError('Could not preserve original age values for every cleaned order.')
ps1_result['Delivery_person_Age_original'] = aligned_source_age.to_numpy()
source_age_text = aligned_source_age.astype('string').str.strip()
source_age_missing_tokens = source_age_text.str.casefold().isin({'', 'nan', 'null', 'none', 'na', 'n/a'})
source_age_numeric = pd.to_numeric(source_age_text.mask(source_age_missing_tokens, pd.NA), errors='coerce')
source_age_invalid = source_age_numeric.notna() & ~source_age_numeric.between(AGE_MIN, AGE_MAX, inclusive='both')
source_age_unparseable = ~source_age_missing_tokens & source_age_numeric.isna()
ps1_result['Delivery_person_Age_invalid'] = source_age_invalid.to_numpy(dtype=bool)
ps1_result['Delivery_person_Age_unparseable'] = source_age_unparseable.to_numpy(dtype=bool)
ps1_result['Vehicle_condition_original'] = vehicle_original
ps1_result['Vehicle_condition_invalid'] = vehicle_invalid.astype(bool)
ps1_result['Vehicle_condition'] = vehicle_original.mask(vehicle_invalid)

# Report the full value distribution before applying the validity rule.
vehicle_counts = vehicle_original.value_counts(dropna=False).sort_index()
vehicle_report = pd.DataFrame({
    'vehicle_condition_original': vehicle_counts.index.astype('string'),
    'record_count': vehicle_counts.to_numpy(dtype=int),
})
vehicle_report['percent_of_records'] = (vehicle_report['record_count'] / ps1_records_before * 100).round(2)
vehicle_report['valid_documented_value'] = pd.to_numeric(vehicle_report['vehicle_condition_original'], errors='coerce').isin(vehicle_valid_values)
vehicle_report['treatment'] = np.where(
    vehicle_report['valid_documented_value'],
    'Retain value',
    'Preserve in Vehicle_condition_original; flag and set Vehicle_condition missing',
)
vehicle_invalid_count = int(vehicle_invalid.sum())

# Source GPS values help distinguish originally present placeholders from values
# that the prior cleaning stage had already changed to missing.
source_audit['__Restaurant_latitude'] = pd.to_numeric(source_audit['Restaurant_latitude'].str.strip(), errors='coerce')
source_audit['__Restaurant_longitude'] = pd.to_numeric(source_audit['Restaurant_longitude'].str.strip(), errors='coerce')
source_audit['__Delivery_location_latitude'] = pd.to_numeric(source_audit['Delivery_location_latitude'].str.strip(), errors='coerce')
source_audit['__Delivery_location_longitude'] = pd.to_numeric(source_audit['Delivery_location_longitude'].str.strip(), errors='coerce')
source_audit = source_audit.set_index('__order_key')
source_geo = source_audit.reindex(ps1_ids_before.to_numpy())
source_geo.index = ps1_result.index

coordinate_pairs = {
    'restaurant': ('Restaurant_latitude', 'Restaurant_longitude'),
    'delivery': ('Delivery_location_latitude', 'Delivery_location_longitude'),
}
india_bounds = {
    'latitude_min': 6.5, 'latitude_max': 37.5,
    'longitude_min': 68.0, 'longitude_max': 98.0,
}
ps1_gps_flags = {}
for location, (latitude_column, longitude_column) in coordinate_pairs.items():
    latitude = pd.to_numeric(ps1_result[latitude_column], errors='coerce')
    longitude = pd.to_numeric(ps1_result[longitude_column], errors='coerce')
    pair_missing = latitude.isna() | longitude.isna()
    pair_zero = latitude.eq(0) | longitude.eq(0)
    pair_placeholder = (
        latitude.notna() & longitude.notna()
        & latitude.abs().lt(1) & longitude.abs().lt(1)
        & np.isclose(latitude.fillna(np.inf), longitude.fillna(-np.inf))
    )
    pair_global_out_of_range = (
        (latitude.notna() & ~latitude.between(-90, 90))
        | (longitude.notna() & ~longitude.between(-180, 180))
    )
    pair_outside_india = (
        (latitude.notna() & ~latitude.between(india_bounds['latitude_min'], india_bounds['latitude_max']))
        | (longitude.notna() & ~longitude.between(india_bounds['longitude_min'], india_bounds['longitude_max']))
    )
    source_latitude = source_geo[f'__{latitude_column}']
    source_longitude = source_geo[f'__{longitude_column}']
    source_zero = source_latitude.eq(0) | source_longitude.eq(0)
    source_placeholder = (
        source_latitude.notna() & source_longitude.notna()
        & source_latitude.abs().lt(1) & source_longitude.abs().lt(1)
        & np.isclose(source_latitude.fillna(np.inf), source_longitude.fillna(-np.inf))
    )
    pair_invalid = pair_missing | pair_zero | pair_placeholder | pair_global_out_of_range | pair_outside_india

    ps1_result[f'{location}_gps_missing'] = pair_missing.astype(bool)
    ps1_result[f'{location}_gps_zero'] = pair_zero.astype(bool)
    ps1_result[f'{location}_gps_placeholder'] = (pair_placeholder | source_placeholder).astype(bool)
    ps1_result[f'{location}_gps_out_of_global_range'] = pair_global_out_of_range.astype(bool)
    ps1_result[f'{location}_gps_outside_india'] = pair_outside_india.astype(bool)
    ps1_result[f'{location}_gps_invalid'] = pair_invalid.astype(bool)
    ps1_result[f'{location}_gps_valid'] = (~pair_invalid).astype(bool)
    ps1_result[f'{location}_source_zero_placeholder'] = source_zero.astype(bool)
    ps1_gps_flags[location] = {
        'missing': pair_missing,
        'zero': pair_zero,
        'placeholder': pair_placeholder | source_placeholder,
        'global_out_of_range': pair_global_out_of_range,
        'outside_india': pair_outside_india,
        'invalid': pair_invalid,
    }

ps1_gps_invalid = ps1_gps_flags['restaurant']['invalid'] | ps1_gps_flags['delivery']['invalid']
ps1_gps_valid = ~ps1_gps_invalid
ps1_gps_missing = ps1_gps_flags['restaurant']['missing'] | ps1_gps_flags['delivery']['missing']
source_restaurant_zero_rows = source_geo['__Restaurant_latitude'].eq(0) & source_geo['__Restaurant_longitude'].eq(0)
source_delivery_near_origin = (
    source_geo['__Delivery_location_latitude'].notna()
    & source_geo['__Delivery_location_longitude'].notna()
    & source_geo['__Delivery_location_latitude'].abs().lt(1)
    & source_geo['__Delivery_location_longitude'].abs().lt(1)
    & np.isclose(
        source_geo['__Delivery_location_latitude'].fillna(np.inf),
        source_geo['__Delivery_location_longitude'].fillna(-np.inf),
    )
)
source_delivery_001 = (
    source_geo['__Delivery_location_latitude'].eq(0.01)
    & source_geo['__Delivery_location_longitude'].eq(0.01)
)
ps1_result['gps_invalid_any'] = ps1_gps_invalid.astype(bool)
ps1_result['gps_geographic_eligible'] = ps1_gps_valid.astype(bool)
ps1_result['gps_missing_any'] = ps1_gps_missing.astype(bool)
ps1_result['gps_zero_or_placeholder_source'] = (source_restaurant_zero_rows | source_delivery_near_origin).astype(bool)

# Compute a safe Haversine distance only for complete, plausible coordinate pairs.
latitude_restaurant = pd.to_numeric(ps1_result['Restaurant_latitude'], errors='coerce')
longitude_restaurant = pd.to_numeric(ps1_result['Restaurant_longitude'], errors='coerce')
latitude_delivery = pd.to_numeric(ps1_result['Delivery_location_latitude'], errors='coerce')
longitude_delivery = pd.to_numeric(ps1_result['Delivery_location_longitude'], errors='coerce')
distance_km = pd.Series(np.nan, index=ps1_result.index, dtype='float64')
if ps1_gps_valid.any():
    lat1 = np.radians(latitude_restaurant.loc[ps1_gps_valid].to_numpy(dtype=float))
    lon1 = np.radians(longitude_restaurant.loc[ps1_gps_valid].to_numpy(dtype=float))
    lat2 = np.radians(latitude_delivery.loc[ps1_gps_valid].to_numpy(dtype=float))
    lon2 = np.radians(longitude_delivery.loc[ps1_gps_valid].to_numpy(dtype=float))
    delta_latitude = lat2 - lat1
    delta_longitude = lon2 - lon1
    haversine_a = (
        np.sin(delta_latitude / 2) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(delta_longitude / 2) ** 2
    )
    haversine_a = np.clip(haversine_a, 0.0, 1.0)
    distance_km.loc[ps1_gps_valid] = 6371.0088 * 2 * np.arcsin(np.sqrt(haversine_a))
ps1_result['Distance_km'] = distance_km

distance_success = ps1_gps_valid & ps1_result['Distance_km'].notna()
distance_values = ps1_result.loc[distance_success, 'Distance_km']
if distance_values.empty:
    distance_upper_fence = np.nan
    unusual_distance = pd.Series(False, index=ps1_result.index)
else:
    distance_q1 = float(distance_values.quantile(0.25))
    distance_q3 = float(distance_values.quantile(0.75))
    distance_iqr = distance_q3 - distance_q1
    distance_upper_fence = distance_q3 + 1.5 * distance_iqr
    unusual_distance = ps1_result['Distance_km'].gt(distance_upper_fence).fillna(False)
ps1_result['distance_unusual_iqr'] = unusual_distance.astype(bool)

# Counts include source placeholders even though the earlier cleaning converted them to missing.
coordinate_report_rows = []
def add_gps_metric(metric, count, detail, treatment):
    coordinate_report_rows.append({
        'metric': metric,
        'record_count': int(count),
        'detail': detail,
        'treatment_or_interpretation': treatment,
    })

add_gps_metric('Total records', ps1_records_before, 'Rows in existing final dataset.', 'Retain all records.')
add_gps_metric('Missing GPS records', int(ps1_gps_missing.sum()), 'At least one restaurant or delivery coordinate pair is missing.', 'Retain; exclude from geographic calculations.')
add_gps_metric('Restaurant (0,0) source placeholders', int(source_restaurant_zero_rows.sum()), 'Observed in the untouched source before earlier cleaning.', 'Keep source evidence flag; do not impute.')
add_gps_metric('Delivery near-origin source pattern', int(source_delivery_near_origin.sum()), 'Both delivery coordinates are equal and below one degree in absolute value; this broad pattern includes (0.01, 0.01).', 'Keep source evidence flag; do not impute.')
add_gps_metric('Delivery (0.01, 0.01) source placeholders', int(source_delivery_001.sum()), 'Exact paired placeholder values in the untouched source.', 'Keep source evidence flag; do not impute.')
add_gps_metric('Missing GPS coordinate cells', int(ps1_result[[c for pair in coordinate_pairs.values() for c in pair]].isna().sum().sum()), 'Missing values across the four coordinate columns.', 'Retain missing; do not impute.')
for coordinate_column in [c for pair in coordinate_pairs.values() for c in pair]:
    source_values = source_geo[f'__{coordinate_column}']
    current_values = pd.to_numeric(ps1_result[coordinate_column], errors='coerce')
    coordinate_is_latitude = coordinate_column.endswith('latitude')
    global_low, global_high = (-90, 90) if coordinate_is_latitude else (-180, 180)
    india_low, india_high = (india_bounds['latitude_min'], india_bounds['latitude_max']) if coordinate_is_latitude else (india_bounds['longitude_min'], india_bounds['longitude_max'])
    add_gps_metric(
        f'{coordinate_column} observed source range', int(source_values.notna().sum()),
        f'min={source_values.min():g}, max={source_values.max():g}; broad India bounds={india_low:g}..{india_high:g}.',
        'Validate against global limits and broad India screening bounds; do not impute.',
    )
    add_gps_metric(
        f'{coordinate_column} source zeros', int(source_values.eq(0).sum()),
        'Count of source rows where this coordinate equals zero.',
        'Flag coordinate pair; retain row and do not impute.',
    )
    add_gps_metric(
        f'{coordinate_column} current missing values', int(current_values.isna().sum()),
        'Missing coordinate values in the current final dataset.',
        'Retain missing; exclude the coordinate pair from geography.',
    )
    add_gps_metric(
        f'{coordinate_column} source out of global range', int((source_values.notna() & ~source_values.between(global_low, global_high)).sum()),
        f'Outside legal coordinate range {global_low}..{global_high}.',
        'Flag pair invalid; retain row.',
    )
    add_gps_metric(
        f'{coordinate_column} source outside India bounds', int((source_values.notna() & ~source_values.between(india_low, india_high)).sum()),
        f'Outside broad India screen {india_low}..{india_high}.',
        'Flag pair invalid; retain recorded coordinate and row.',
    )
add_gps_metric('Current zero-coordinate records', int((ps1_result[[c for pair in coordinate_pairs.values() for c in pair]].eq(0).any(axis=1)).sum()), 'Any coordinate is exactly zero in the current final dataset.', 'Flag and exclude geographically.')
add_gps_metric('Out-of-global-range GPS records', int((ps1_gps_flags['restaurant']['global_out_of_range'] | ps1_gps_flags['delivery']['global_out_of_range']).sum()), 'Latitude outside [-90, 90] or longitude outside [-180, 180].', 'Flag and exclude geographically.')
add_gps_metric('Outside broad India bounds records', int((ps1_gps_flags['restaurant']['outside_india'] | ps1_gps_flags['delivery']['outside_india']).sum()), 'At least one coordinate pair lies outside 6.5 to 37.5 degrees N, 68 to 98 degrees E.', 'Flag and exclude geographically; retain recorded values.')
add_gps_metric('Invalid GPS records', int(ps1_gps_invalid.sum()), 'Unique records with missing, zero, placeholder, out-of-range, or outside-India coordinates.', 'Preserve rows and coordinates; do not use for distance/geographic clustering.')
add_gps_metric('Valid GPS records', int(ps1_gps_valid.sum()), 'Both restaurant and delivery pairs are present and pass checks.', 'Eligible for distance/geographic analysis.')
add_gps_metric('Distance calculation successes', int(distance_success.sum()), 'Valid GPS rows with a finite computed Haversine distance.', 'Distance stored in Distance_km.')
add_gps_metric('Valid-GPS rows without distance', int((ps1_gps_valid & ~distance_success).sum()), 'A valid GPS row unexpectedly lacks distance.', 'Validation must be zero; investigate if nonzero.')
add_gps_metric('Non-negative finite calculated distances', int((distance_values.ge(0) & np.isfinite(distance_values)).sum()), 'Checked all calculated distances.', 'Validation must equal distance calculation successes.')
add_gps_metric('Unusual distance records', int(unusual_distance.sum()), f'Distance above 1.5 times the IQR upper fence ({distance_upper_fence:.3f} km).', 'Flag for investigation; do not delete.')
add_gps_metric('Records removed', 0, 'No rows removed by this final audit.', 'None.')
gps_validation_report = pd.DataFrame(coordinate_report_rows)

vehicle_report_summary = pd.DataFrame([
    {'vehicle_condition_original': 'INVALID_TOTAL', 'record_count': vehicle_invalid_count,
     'percent_of_records': round(vehicle_invalid_count / ps1_records_before * 100, 2),
     'valid_documented_value': False,
     'treatment': 'Preserve original; flag and convert Vehicle_condition to missing'},
    {'vehicle_condition_original': 'RECORDS_REMOVED', 'record_count': 0,
     'percent_of_records': 0.0, 'valid_documented_value': np.nan, 'treatment': 'None'},
])
vehicle_validation_report = pd.concat([vehicle_report, vehicle_report_summary], ignore_index=True)

# Validated invariants: no rows lost, no duplicate IDs, prior treatments intact, and target unchanged.
if len(ps1_result) != ps1_records_before:
    raise AssertionError('Record count changed during final audit.')
if ps1_result['ID'].astype('string').str.strip().duplicated().any():
    raise AssertionError('Duplicate order IDs found after final audit.')
if int(ps1_result.duplicated().sum()) != ps1_full_duplicates_before:
    raise AssertionError('Exact duplicate-row count changed during final audit.')
if not np.array_equal(ps1_result[TARGET_COLUMN].to_numpy(dtype=float), ps1_target_before.to_numpy(dtype=float)):
    raise AssertionError('The delivery-time target changed during final audit.')
if not ps1_result.loc[~ps1_result['gps_geographic_eligible'], 'Distance_km'].isna().all():
    raise AssertionError('Invalid GPS rows must not have a calculated distance.')
if not (ps1_result.loc[ps1_result['gps_geographic_eligible'], 'Distance_km'].notna().all()):
    raise AssertionError('Every valid GPS row must have a distance.')
calculated_distances = ps1_result.loc[ps1_result['gps_geographic_eligible'], 'Distance_km'].to_numpy(dtype=float)
if not (np.isfinite(calculated_distances).all() and (calculated_distances >= 0).all()):
    raise AssertionError('Calculated distances must be finite and non-negative.')
if source_age_invalid.sum() != int(ps1_result['Delivery_person_Age_invalid'].sum()):
    raise AssertionError('Age invalid flags do not match the inclusive 18-80 validation rule.')
if source_age_unparseable.sum() != int(ps1_result['Delivery_person_Age_unparseable'].sum()):
    raise AssertionError('Age unparseable flags do not match original source values.')
if not ps1_result['Delivery_person_Age_original'].astype('string').reset_index(drop=True).equals(aligned_source_age.astype('string').reset_index(drop=True)):
    raise AssertionError('Original age values were not preserved by order ID.')
if vehicle_invalid_count != int(ps1_result['Vehicle_condition_invalid'].sum()):
    raise AssertionError('Vehicle-condition invalid flags do not match the audit count.')
if not ps1_result.loc[ps1_result['Vehicle_condition_invalid'], 'Vehicle_condition'].isna().all():
    raise AssertionError('Invalid vehicle values must be missing in the validated column.')
if not ps1_result.loc[ps1_result['Vehicle_condition_invalid'], 'Vehicle_condition_original'].eq(3).all():
    raise AssertionError('The original invalid vehicle values must remain auditable.')

# All prior missing-value decisions remain unchanged except the explicitly invalid vehicle category.
prior_imputed_numeric = ['Delivery_person_Age', 'Delivery_person_Ratings', 'multiple_deliveries']
prior_filled_categories = ['Weatherconditions', 'Road_traffic_density', 'Festival', 'City']
if ps1_result[prior_imputed_numeric].isna().any().any() or ps1_result[prior_filled_categories].isna().any().any():
    raise AssertionError('A previous missing-value treatment was not preserved.')
for timestamp_column in ['Order_Date', 'Time_Orderd', 'Time_Order_picked']:
    if not ps1_result[timestamp_column].isna().equals(ps1_before[timestamp_column].isna()):
        raise AssertionError(f'Missing timestamp values changed in {timestamp_column}.')

ps1_output_dir.mkdir(parents=True, exist_ok=True)
try:
    ps1_result.to_csv(ps1_input_path, index=False, na_rep='')
    gps_validation_report.to_csv(ps1_gps_report_path, index=False)
    vehicle_validation_report.to_csv(ps1_vehicle_report_path, index=False)
except OSError as exc:
    raise RuntimeError(f'Could not save final validation outputs: {exc}') from exc

vehicle_distribution_text = '\n'.join(
    f"- {value}: {count:,} ({count / ps1_records_before * 100:.2f}%)"
    for value, count in vehicle_original.value_counts(dropna=False).sort_index().items()
)
summary_lines = [
    'Quick-Commerce Delivery Intelligence - PS1 Final GPS and Vehicle Data Validation',
    '',
    f'Total records before / after: {ps1_records_before:,} / {len(ps1_result):,}',
    f'Age validation range: {AGE_MIN}-{AGE_MAX} years, inclusive.',
    f'Invalid / unparseable age values: {int(source_age_invalid.sum()):,} / {int(source_age_unparseable.sum()):,}; original values and flags preserved.',
    AGE_EXPLANATION,
    f'Exact duplicate rows before / after: {ps1_full_duplicates_before:,} / {int(ps1_result.duplicated().sum()):,}',
    f'Records removed: 0 (none)',
    '',
    'Vehicle_condition value counts before validation:',
    vehicle_distribution_text,
    f'Invalid vehicle-condition records (value outside documented 0, 1, 2): {vehicle_invalid_count:,}',
    f'Invalid vehicle values converted to missing, original preserved: {vehicle_invalid_count:,}',
    '',
    'GPS validation:',
    f'Missing GPS records: {int(ps1_gps_missing.sum()):,}',
    f'Source restaurant (0,0) placeholder records: {int(source_restaurant_zero_rows.sum()):,}',
    f'Source delivery near-origin pattern records: {int(source_delivery_near_origin.sum()):,}',
    f'Source delivery (0.01, 0.01) placeholder records: {int(source_delivery_001.sum()):,}',
    f'Missing GPS coordinate cells: {int(ps1_result[[c for pair in coordinate_pairs.values() for c in pair]].isna().sum().sum()):,}',
    f'Outside broad India bounds records: {int((ps1_gps_flags["restaurant"]["outside_india"] | ps1_gps_flags["delivery"]["outside_india"]).sum()):,}',
    f'Invalid GPS records: {int(ps1_gps_invalid.sum()):,}',
    f'Valid GPS records: {int(ps1_gps_valid.sum()):,}',
    f'Distance calculation successes: {int(distance_success.sum()):,}',
    f'Unusual distance records (upper 1.5 times the IQR fence {distance_upper_fence:.3f} km): {int(unusual_distance.sum()):,}',
    '',
    'Distance validation: invalid GPS rows have no Distance_km; calculated distances are finite and non-negative.',
    'GPS coordinates were not imputed. Invalid GPS records remain available for non-geographic analyses.',
    'Target validation: Time_taken(min) matches the original source by order ID and is unchanged.',
    'Previous missing-value treatments (median numeric fills, Unknown categories, timestamp gaps/flags) remain intact.',
    'Vehicle documentation reference: https://github.com/Chandansahu18/food-delivery-operations-analytics-dashboard/blob/42683f1123fd417ceec94e5a749aee0293ee42c0/docs/data_dictionary.md',
]
try:
    ps1_summary_path.write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
except OSError as exc:
    raise RuntimeError(f'Could not save final PS1 summary: {exc}') from exc

# Reload saved artifacts to ensure the persisted files match the validated in-memory result.
ps1_saved = pd.read_csv(ps1_input_path, keep_default_na=True)
if len(ps1_saved) != ps1_records_before or int(ps1_saved.duplicated().sum()) != ps1_full_duplicates_before:
    raise AssertionError('Saved final dataset failed row-count or duplicate validation.')
ps1_saved_age = pd.read_csv(ps1_input_path, dtype={'Delivery_person_Age_original': 'string'}, keep_default_na=False)['Delivery_person_Age_original']
if ps1_saved_age.tolist() != aligned_source_age.astype('string').tolist():
    raise AssertionError('Saved original age values differ from source values by order ID.')
if not ps1_saved['Delivery_person_Age_invalid'].to_numpy(dtype=bool).tolist() == source_age_invalid.to_numpy(dtype=bool).tolist():
    raise AssertionError('Saved age-invalid flags differ from the 18-80 source validation.')
if not np.array_equal(ps1_saved[TARGET_COLUMN].to_numpy(dtype=float), ps1_target_before.to_numpy(dtype=float)):
    raise AssertionError('Saved target differs from the pre-audit target.')
if not ps1_saved.loc[~ps1_saved['gps_geographic_eligible'], 'Distance_km'].isna().all():
    raise AssertionError('Saved invalid GPS rows contain distances.')

print('VEHICLE CONDITION COUNTS BEFORE TREATMENT')
display(vehicle_report)
print('GPS VALIDATION SUMMARY')
display(gps_validation_report)
print('\n'.join(summary_lines))
print('Saved:', ps1_input_path, ps1_gps_report_path, ps1_vehicle_report_path, ps1_summary_path)

VEHICLE CONDITION COUNTS BEFORE TREATMENT


,vehicle_condition_original,record_count,percent_of_records,valid_documented_value,treatment
0,0,15009,32.92,True,Retain value
1,1,15030,32.97,True,Retain value
2,2,15034,32.97,True,Retain value
3,3,520,1.14,False,Preserve in Vehicle_condition_original; flag and set Vehicle_condition missing


GPS VALIDATION SUMMARY


,metric,record_count,detail,treatment_or_interpretation
0,Total records,45593,Rows in existing final dataset.,Retain all records.
1,Missing GPS records,3640,At least one restaurant or delivery coordinate pair is missing.,Retain; exclude from geographic calculations.
2,"Restaurant (0,0) source placeholders",3640,Observed in the untouched source before earlier cleaning.,Keep source evidence flag; do not impute.
3,Delivery near-origin source pattern,3640,"Both delivery coordinates are equal and below one degree in absolute value; this broad pattern includes (0.01, 0.01).",Keep source evidence flag; do not impute.
4,"Delivery (0.01, 0.01) source placeholders",327,Exact paired placeholder values in the untouched source.,Keep source evidence flag; do not impute.
5,Missing GPS coordinate cells,14560,Missing values across the four coordinate columns.,Retain missing; do not impute.
6,Restaurant_latitude observed source range,45593,"min=-30.9056, max=30.9141; broad India bounds=6.5..37.5.",Validate against global limits and broad India screening bounds; do not impute.
7,Restaurant_latitude source zeros,3640,Count of source rows where this coordinate equals zero.,Flag coordinate pair; retain row and do not impute.
8,Restaurant_latitude current missing values,3640,Missing coordinate values in the current final dataset.,Retain missing; exclude the coordinate pair from geography.
9,Restaurant_latitude source out of global range,0,Outside legal coordinate range -90..90.,Flag pair invalid; retain row.


Quick-Commerce Delivery Intelligence - PS1 Final GPS and Vehicle Data Validation

Total records before / after: 45,593 / 45,593
Age validation range: 18-80 years, inclusive.
Invalid / unparseable age values: 38 / 0; original values and flags preserved.
For consistency across the project, rider age values are considered valid for analysis when they fall between 18 and 80 years, inclusive. This is a data-validation rule and does not represent rider professional experience.
Exact duplicate rows before / after: 0 / 0
Records removed: 0 (none)

Vehicle_condition value counts before validation:
- 0: 15,009 (32.92%)
- 1: 15,030 (32.97%)
- 2: 15,034 (32.97%)
- 3: 520 (1.14%)
Invalid vehicle-condition records (value outside documented 0, 1, 2): 520
Invalid vehicle values converted to missing, original preserved: 520

GPS validation:
Missing GPS records: 3,640
Source restaurant (0,0) placeholder records: 3,640
Source delivery near-origin pattern records: 3,640
Source delivery (0.01, 0.01) placeh